In [ ]:
!pip install gymnasium ale-py autorom opencv-python tensorflow

In [ ]:
!AutoROM --accept-license

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms

Existing ROMs will be overwritten.


In [ ]:
import ale_py
import gymnasium as gym
import time
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from collections import deque
import random

print("Imports done")

Imports done


In [ ]:
""" LOAD SAVED MODEL """

# Load the model
online_net = tf.keras.models.load_model("kingkong_dqn.keras")
print("Model loaded from kingkong_dqn.keras")

Model loaded from kingkong_dqn.keras


In [ ]:
""" TRAIN NEW MODEL """

gym.register_envs(ale_py)

env = gym.make("ALE/KingKong-v5", render_mode="rgb_array")
obs, info = env.reset()

# ── Hyperparameters ────────────────────────────────────────────────────────────
INPUT_SHAPE     = (84, 84, 1)
NUM_ACTIONS     = int(env.action_space.n)

TRAIN_SECONDS   = 300          # How long to train (seconds)
EVAL_SECONDS    = 30           # How long each evaluation window lasts
EVAL_EVERY      = 60           # Run an evaluation every N training seconds

GAMMA           = 0.99         # Discount factor
EPSILON_START   = 1.0          # Initial exploration rate
EPSILON_MIN     = 0.05         # Minimum exploration rate
EPSILON_DECAY   = 0.995        # Multiplicative decay applied each episode

BATCH_SIZE      = 32
REPLAY_CAPACITY = 20_000       # Experience replay buffer size
MIN_REPLAY_SIZE = 1_000        # Don't train until buffer has this many samples
LEARNING_RATE   = 5e-5
TARGET_UPDATE_EVERY = 500      # Steps between target-network syncs


# ── Networks ───────────────────────────────────────────────────────────────────
def build_q_network(input_shape, num_actions):
    """
    A small CNN Q-network.
    Outputs Q(s, a) for every action a simultaneously.
    """
    inputs = keras.Input(shape=input_shape)
    x = keras.layers.Conv2D(32, 8, strides=4, activation="relu")(inputs)
    x = keras.layers.Conv2D(64, 4, strides=2, activation="relu")(x)
    x = keras.layers.Conv2D(64, 3, strides=1, activation="relu")(x)  # NEW layer
    x = keras.layers.Flatten()(x)
    x = keras.layers.Dense(256, activation="relu")(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_actions, activation=None)(x)
    return keras.Model(inputs, outputs)

# Online network  — trained every step
online_net = build_q_network(INPUT_SHAPE, NUM_ACTIONS)
# Target network — updated periodically for stable TD targets
target_net = build_q_network(INPUT_SHAPE, NUM_ACTIONS)
target_net.set_weights(online_net.get_weights())

optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)
online_net.summary()
print("Networks created")


# ── Helpers ────────────────────────────────────────────────────────────────────
def preprocess_frame(frame):
    gray  = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return (small / 255.0).astype(np.float32)

def select_action(frame, epsilon):
    if np.random.rand() < epsilon:
        return env.action_space.sample()
    processed = preprocess_frame(frame)
    tensor    = processed[np.newaxis, ..., np.newaxis]          # (1,84,84,1)
    logits    = online_net(tensor, training=False)
    return int(tf.argmax(logits[0]).numpy())

replay_buffer = deque(maxlen=REPLAY_CAPACITY)

def store_transition(state, action, reward, next_state, done):
    replay_buffer.append((state, action, reward, next_state, done))

@tf.function
def train_step(states, actions, rewards, next_states, dones):
    # TD target: r + γ · max_a Q_target(s', a)   (0 if terminal)
    next_q   = target_net(next_states, training=False)
    max_next = tf.reduce_max(next_q, axis=1)
    targets  = rewards + GAMMA * max_next * (1.0 - dones)

    with tf.GradientTape() as tape:
        q_values = online_net(states, training=True)
        # Select only the Q-value for the action that was actually taken
        actions_oh = tf.one_hot(actions, NUM_ACTIONS)
        predicted  = tf.reduce_sum(q_values * actions_oh, axis=1)
        loss = tf.reduce_mean(tf.square(targets - predicted))   # MSE

    grads = tape.gradient(loss, online_net.trainable_variables)
    optimizer.apply_gradients(zip(grads, online_net.trainable_variables))
    return loss

def sample_and_train():
    batch = random.sample(replay_buffer, BATCH_SIZE)
    states, actions, rewards, next_states, dones = zip(*batch)

    states      = tf.constant(np.array(states),      dtype=tf.float32)
    actions     = tf.constant(np.array(actions),     dtype=tf.int32)
    rewards     = tf.constant(np.array(rewards),     dtype=tf.float32)
    next_states = tf.constant(np.array(next_states), dtype=tf.float32)
    dones       = tf.constant(np.array(dones),       dtype=tf.float32)

    return train_step(states, actions, rewards, next_states, dones)


# ── Evaluation ─────────────────────────────────────────────────────────────────
def evaluate(eval_seconds=EVAL_SECONDS):
    """
    Run the greedy policy (epsilon=0) for a fixed wall-clock window.
    Returns total reward accumulated across however many episodes fit.
    """
    eval_env = gym.make("ALE/KingKong-v5", render_mode="rgb_array")
    frame, _ = eval_env.reset()
    total_reward = 0.0
    deadline     = time.time() + eval_seconds

    while time.time() < deadline:
        processed = preprocess_frame(eval_env.render())
        tensor    = processed[np.newaxis, ..., np.newaxis]
        logits    = online_net(tensor, training=False)
        action    = int(tf.argmax(logits[0]).numpy())

        _, reward, terminated, truncated, _ = eval_env.step(action)
            # --- Custom Reward System ---
    reward -= 0.01

    if terminated:
        reward -= 5

# OPTIONAL (only if you can track progress)
# Example placeholder:
# if progressed_forward:
#     reward += 10
        total_reward += reward

        if terminated or truncated:
            frame, _ = eval_env.reset()     # Keep playing until time is up

    eval_env.close()
    return total_reward


# ── Training loop ──────────────────────────────────────────────────────────────
print("Filling replay buffer …")
frame       = env.render()
epsilon     = EPSILON_START
total_steps = 0
episode_num = 0

eval_scores  = []   # (wall_time, score) checkpoints
train_start  = time.time()
last_eval_at = train_start

# ── Phase 1: pre-fill replay buffer with random experience ─────────────────────
prefill_obs, _ = env.reset()
while len(replay_buffer) < MIN_REPLAY_SIZE:
    f = env.render()
    a = env.action_space.sample()
    _, r, term, trunc, _ = env.step(a)
    s  = preprocess_frame(f)[..., np.newaxis]
    f2 = env.render()
    s2 = preprocess_frame(f2)[..., np.newaxis]
    store_transition(s, a, r, s2, False)
    if term or trunc:
        env.reset()

print(f"Replay buffer filled ({len(replay_buffer)} transitions). Training …")

# ── Phase 2: main train + periodically-evaluate loop ──────────────────────────
obs, _ = env.reset()

while time.time() - train_start < TRAIN_SECONDS:
    frame  = env.render()
    state  = preprocess_frame(frame)[..., np.newaxis]   # (84,84,1)

    action = select_action(frame, epsilon)
    _, reward, terminated, truncated, _ = env.step(action)

    next_frame = env.render()
    next_state = preprocess_frame(next_frame)[..., np.newaxis]

    # Treat time-limit truncation as non-terminal for bootstrapping
    store_transition(state, action, reward, next_state,
                     float(terminated and not truncated))

    loss = sample_and_train()
    total_steps += 1

    # Sync target network
    if total_steps % TARGET_UPDATE_EVERY == 0:
        target_net.set_weights(online_net.get_weights())
        print(f"  [step {total_steps}] target network updated")

    if terminated or truncated:
        episode_num += 1
        epsilon = max(EPSILON_MIN, epsilon * EPSILON_DECAY)
        obs, _ = env.reset()

    # ── Periodic evaluation checkpoint ────────────────────────────────────────
    now = time.time()
    if now - last_eval_at >= EVAL_EVERY:
        elapsed = now - train_start
        print(f"\n{'─'*50}")
        print(f"Evaluating at t={elapsed:.0f}s  (ε={epsilon:.3f}, "
              f"steps={total_steps}, episodes={episode_num}) …")
        score = evaluate(EVAL_SECONDS)
        eval_scores.append((elapsed, score))
        print(f"  Eval score ({EVAL_SECONDS}s window): {score:.1f}")
        print(f"{'─'*50}\n")
        last_eval_at = time.time()   # Reset clock after eval overhead

print("\nTraining complete.")
print("\nPerformance over time:")
print(f"  {'Time (s)':>10}  {'Score':>10}")
for t, s in eval_scores:
    print(f"  {t:>10.0f}  {s:>10.1f}")

env.close()
cv2.destroyAllWindows()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 84, 84, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 20, 20, 32)     │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 9, 9, 64)       │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 7, 7, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       803,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 876,454 (3.34 MB)

 Trainable params: 876,454 (3.34 MB)

 Non-trainable params: 0 (0.00 B)

Networks created
Filling replay buffer …
Replay buffer filled (1000 transitions). Training …
  [step 500] target network updated

──────────────────────────────────────────────────
Evaluating at t=60s  (ε=1.000, steps=995, episodes=0) …
  Eval score (30s window): -5.0
──────────────────────────────────────────────────

  [step 1000] target network updated
  [step 1500] target network updated

──────────────────────────────────────────────────
Evaluating at t=150s  (ε=0.995, steps=1999, episodes=1) …
  Eval score (30s window): 0.0
──────────────────────────────────────────────────

  [step 2000] target network updated
  [step 2500] target network updated
  [step 3000] target network updated

──────────────────────────────────────────────────
Evaluating at t=241s  (ε=0.995, steps=3036, episodes=1) …
  Eval score (30s window): -5.0
──────────────────────────────────────────────────

  [step 3500] target network updated

Training complete.

Performance over time:
    Time (s)       Score
 

In [ ]:
""" TEST PERFORMANCE """

gym.register_envs(ale_py)

def preprocess_obs(obs):
    gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
    small = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return (small / 255.0).astype(np.float32)

eval_env = gym.make("ALE/KingKong-v5", render_mode="human")
obs, _ = eval_env.reset()

deadline = time.time() + 60

while time.time() < deadline:
    tensor = preprocess_obs(obs)[np.newaxis, ..., np.newaxis]
    action = int(tf.argmax(online_net(tensor, training=False)[0]).numpy())

    obs, reward, terminated, truncated, _ = eval_env.step(action)

    if terminated or truncated:
        obs, _ = eval_env.reset()

eval_env.close()

In [ ]:
online_net.save("model2.keras")
print("Model 2 saved to model2.keras")

Model 2 saved to model2.keras
